In [ ]:
from pathlib import Path


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in (current, *current.parents):
        if (path / "data").exists():
            return path
    return current


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"

import gc

import numpy as np
import pandas as pd

INPUT_DIR = DATA_ROOT / "data_train" / "oto_vwap_5min"
OUTPUT_DIR = DATA_ROOT / "data_train" / "oto_vwap_5min_zscore"
KEYWORD = "oto"
SUFFIX = "_rank_zscore"
MODE = "add"
CODE_PREFIXES = ("0", "3", "6")


def rank_then_zscore(values):
    x = values.astype(float, copy=True)
    x[np.isinf(x)] = np.nan
    ranks = pd.Series(x).rank(method="average", na_option="keep").to_numpy(dtype=float)
    mean = np.nanmean(ranks)
    std = np.nanstd(ranks)
    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        return np.full_like(ranks, np.nan, dtype=float)
    return (ranks - mean) / std


def code_series(series):
    return series.astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)


def filter_codes(frame):
    code_col = "code" if "code" in frame.columns else "Code"
    if code_col not in frame.columns:
        raise ValueError("input parquet must contain a code column")
    codes = code_series(frame[code_col])
    mask = pd.Series(False, index=frame.index)
    for prefix in CODE_PREFIXES:
        mask |= codes.str.startswith(prefix)
    out = frame.loc[mask].copy()
    out[code_col] = codes.loc[mask].values
    return out


def process_file(in_path):
    out_path = OUTPUT_DIR / in_path.name
    frame = pd.read_parquet(in_path)
    frame = filter_codes(frame)
    columns = [col for col in frame.columns if KEYWORD in col]
    mode = MODE.strip().lower()
    if mode not in {"add", "overwrite"}:
        raise ValueError("MODE must be add or overwrite")
    for col in columns:
        values = pd.to_numeric(frame[col], errors="coerce").to_numpy(dtype=float, copy=False)
        target_col = f"{col}{SUFFIX}" if mode == "add" else col
        frame[target_col] = rank_then_zscore(values)
    if mode == "overwrite":
        frame = frame.rename(columns={col: f"{col}{SUFFIX}" for col in columns})
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(out_path, index=False)
    rows = len(frame)
    del frame
    gc.collect()
    return in_path.name, rows


def main():
    files = sorted(INPUT_DIR.glob("*.parquet"))
    for path in files:
        name, rows = process_file(path)
        print(f"file={name} rows={rows}")
    print(f"done files={len(files)} output={OUTPUT_DIR}")


if __name__ == "__main__":
    main()